# Gene annotation and gene profiling



## The workflow:

*   Bacterial gene prediction: prodigal
*   Generating the gene catalog: cd-hit
*   Building a bowtie2 database from the gene catalog: bowtie2
*   Mapping short reads against the gene catalog: bowtie2
*   Calculating the coverage for each sample

## Setup of the environment



In [2]:
# conda environment
import os,sys
root_dir = "/biodata/resources/day3_lab4"



## Check the installation was successful

In [3]:
! which prodigal
! which samtools
! which bowtie2
! which cd-hit


/opt/conda/bin/prodigal
/opt/conda/bin/samtools
/opt/conda/bin/bowtie2
/opt/conda/bin/cd-hit


## Predict bacterial genes on contigs

In [4]:
import os
root_dir = "/biodata/resources/day3_lab4"
gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call")
contigs_demo_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","contig_demo")
sample_id = "PSMB4MBK"

! mkdir -p {gene_call_dir}
output_gff = os.path.join(gene_call_dir,sample_id+".prodigal.gff")
output_faa = os.path.join(gene_call_dir,sample_id+".prodigal.faa")
output_fna = os.path.join(gene_call_dir,sample_id+".prodigal.fna")
! ls -lh {contigs_demo_dir}/{sample_id}_contigs.fna


-rw-rw-rw- 1 root root 58M Sep 29 15:13 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/contig_demo/PSMB4MBK_contigs.fna


In [ ]:
# run gene prediction
! prodigal -p meta -i {contigs_demo_dir}/{sample_id}_contigs.fna -f gff -o {output_gff} -a {output_faa} -d {output_fna}  > /dev/null 2>&1 # take 7 min for run

-rw-rw-rw- 1 root root 58M Sep 29 15:13 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/contig_demo/PSMB4MBK_contigs.fna


## Overview of the output


In [5]:
demo_gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call_demo")
# pre-calculated files
output_gff = os.path.join(demo_gene_call_dir,sample_id+".prodigal.gff")
output_faa = os.path.join(demo_gene_call_dir,sample_id+".prodigal.faa")
output_fna = os.path.join(demo_gene_call_dir,sample_id+".prodigal.fna")
! head {output_fna}

>k105_1_1 # 3 # 107 # -1 # ID=1_1;partial=10;start_type=ATG;rbs_motif=None;rbs_spacer=None;gc_cont=0.429
ATGGATGAACTGACCGACATTTACAAAAGAATAGAATACCTGCGCAACAATGGCGTAAAGATGAAAGAAA
TTGCCGACCGTGTAGATATGGCACCAAGCGTATTG
>k105_2_1 # 29 # 2416 # -1 # ID=2_1;partial=01;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.371
TCCATTACAGTGCGTAAATTGGGGCGCATGCCTAAAATCGTAAAAGATCCTTATATACAGGCTAGCTATA
AAAAAATTATGGGTAAGCCTTGGTACGATCTTTATTCAGACGAGGATTTAAAATACATTGAAAAGATGAA
AGAAGACCCAACCTTACCTAATGTGATTCCAAGTTTCAGTAATCCAGAATATTATACCTATTTAGGGAAT
ACAGACTGGTTTTCGGAAATATATGATAACACAGGTATAACTCACTCTCATAATTTAAGTTTGTCAGGCG
CTTCCGAAAAGGCTTCTTATTATATAGGTATGGAATATATGCAGGAAAGAGGGCTCTTAAAAATTAATAA
GGACATTATGGATCGTTATAATTTCCGCTCAAAAGTAGATTTCAAAGTAGCCGATTGGTTGACTTTTGGC


### Task: count all complete genes

Incomplete genes are marked by the "partial" field in the header:

*   A complete gene: **partial=00**
*   A gene that is incomplete on the left side: **partial=10**
*   A gene that is incomplete on the right side: **partial=01**
*   A gene that is incomplete on both sides: **partial=11**

Calculate the number of complete genes and the total number of genes.

Hint:


*   **grep** the headers with **partial=00**
*   count the results using the **wc** commend




In [6]:
! grep "partial=00" {output_fna} | wc -l
! grep ">" {output_fna} | wc -l

16961
59579


### Put your solution here [2 min]


In [6]:
# your solution here:


## Generating the gene catalog

A gene catalog consist of all possible, **non-redundent** protein-coding genes

Assuming that we have assembled genes from multiple samples, we are going to generate a gene catalog from these files.

We will start with a merged file with all the **complete** genes from multiple samples

In [7]:
demo_gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog")
merged_fa = os.path.join(demo_gene_call_dir,"merged_genes.fna")
! head {merged_fa}

>CSM5MCXT_k105_1::1::76::309::-
ATGAAAGATATAGAAGTAAACGGCGCACATATAACAGATGAAAGTGCCGAGATTTTGAAA
CAGTGGCAAGTTAAGACGGAACCGGTTTCCGCTTGTTACATCGAAGTTATTGAGGACCTA
ATCGATTTCCTAATAGAGAAAGGAGATGAAAGTACACCAACAAATGAGGTGTTAAGAAGG
ATTCAATTATTACGTATGATGAAAAAAGACATCGAAAAGTTGTCTAATCCTTAA
>CSM5MCXT_k105_1::2::323::697::-
ATGAATACAAATAATCCTGATATTCTATTTTTCGTTAGACGTGAATACGGTGCACCTTCC
ATTGAATTAAGAGCATATAAGGTGGAGAAGGTAAACGAAGAATTTGCTTTCCTCGAACTT
GAACGTTTGCGGTTGGTTGTTTTCTCCGGTGATTTTCAGTCTGTATCACTTCATCACGAG
TACGGTAAAAACAACTGTCTGTATAATAGTGCCAATAATATACCGGATTTGATGAAAGAC


In [8]:
coverage=90
identity=95

! cd-hit-est -i {merged_fa} -aS 0.{coverage} -aL 0.{coverage} -c 0.{identity} -M 0 -r 0 -B 0 -d 0 -o {demo_gene_call_dir}/nr.fa -sc 1;

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:59:25
Command: cd-hit-est -i
         /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/merged_genes.fna
         -aS 0.90 -aL 0.90 -c 0.95 -M 0 -r 0 -B 0 -d 0 -o
         /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa
         -sc 1

Started: Tue Sep 30 21:09:07 2025
                            Output                              
----------------------------------------------------------------
total seq: 21584
longest and shortest : 12213 and 90
Total letters: 18729594
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 22M
Buffer          : 1 X 20M = 20M
Table           : 1 X 17M = 17M
Miscellaneous   : 0M
Total           : 59M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 647106500

comparing sequences from          0  to      21584
..........    10000  finished 

### Overview of the result

There are two major output files:


1.   The representative sequence of each cluster [fasta format]
2.   A cluster file recording the cluster each gene originates from [.clsr]



In [9]:
! head {demo_gene_call_dir}/nr.fa

>CSM5MCXT_k105_1::1::76::309::-
ATGAAAGATATAGAAGTAAACGGCGCACATATAACAGATGAAAGTGCCGAGATTTTGAAA
CAGTGGCAAGTTAAGACGGAACCGGTTTCCGCTTGTTACATCGAAGTTATTGAGGACCTA
ATCGATTTCCTAATAGAGAAAGGAGATGAAAGTACACCAACAAATGAGGTGTTAAGAAGG
ATTCAATTATTACGTATGATGAAAAAAGACATCGAAAAGTTGTCTAATCCTTAA
>CSM5MCXT_k105_1::2::323::697::-
ATGAATACAAATAATCCTGATATTCTATTTTTCGTTAGACGTGAATACGGTGCACCTTCC
ATTGAATTAAGAGCATATAAGGTGGAGAAGGTAAACGAAGAATTTGCTTTCCTCGAACTT
GAACGTTTGCGGTTGGTTGTTTTCTCCGGTGATTTTCAGTCTGTATCACTTCATCACGAG
TACGGTAAAAACAACTGTCTGTATAATAGTGCCAATAATATACCGGATTTGATGAAAGAC


In [10]:
! head {demo_gene_call_dir}/nr.fa.clstr

>Cluster 0
0	117nt, >CSM5MCXT_k105_7934::2::331::447::-... *
1	117nt, >CSM5MCXT_k105_39322::2::174::290::+... at +/97.44%
2	117nt, >CSM5MCXT_k105_41278::23::23895::24011::-... at +/95.73%
3	117nt, >CSM5MCW6_k105_30839::49::69293::69409::-... at +/95.73%
>Cluster 1
0	315nt, >CSM5MCXT_k105_1156::2::349::663::+... *
1	315nt, >CSM5MCXT_k105_14525::2::349::663::+... at +/96.83%
2	315nt, >CSM5MCW6_k105_8476::1::62::376::-... at +/98.10%
>Cluster 2


### Task: count the number of resulting clusters

In [11]:
# your solution here:
! tail {demo_gene_call_dir}/nr.fa.clstr

>Cluster 18987
0	858nt, >CSM5MCXT_k105_6584::2::296::1153::+... *
>Cluster 18988
0	858nt, >CSM5MCXT_k105_16943::6::6402::7259::-... *
>Cluster 18989
0	858nt, >CSM5MCXT_k105_17213::1::53::910::+... *
>Cluster 18990
0	858nt, >CSM5MCXT_k105_14769::5::3299::4156::+... *
>Cluster 18991
0	858nt, >CSM5MCXT_k105_16395::3::2850::3707::-... *


In [12]:
! grep ">Cluster" {demo_gene_call_dir}/nr.fa.clstr | wc -l

18992


## Create a bowtie2 database from the gene catalog



In [13]:
# build bowtie2 database. 1 min for demo
! bowtie2-build {demo_gene_call_dir}/nr.fa {demo_gene_call_dir}/nr.fa_bowtie2DB

Settings:
  Output files: "/biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 4 (one in 16)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local endianness: little
  Sanity checking: disabled
  Assertions: disabled
  Random seed: 0
  Sizeofs: void*:8, int:4, long:8, size_t:8
Input files DNA, FASTA:
  /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa
Building a SMALL index
Reading reference sizes
  Time reading reference sizes: 00:00:00
Calculating joined length
Writing header
Reserving space for joined string
Joining reference sequences
  Time to join reference sequences: 00:00:00
bmax according to bmaxDivN setting: 3984285
Using parameters --bmax 298821

In [14]:
! ls -lh {demo_gene_call_dir}/nr.fa_bowtie2DB*

-rw-r--r-- 1 root root  11M Sep 30 21:09 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.1.bt2
-rw-r--r-- 1 root root 3.8M Sep 30 21:09 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.2.bt2
-rw-r--r-- 1 root root 167K Sep 30 21:09 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.3.bt2
-rw-r--r-- 1 root root 3.8M Sep 30 21:09 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.4.bt2
-rw-r--r-- 1 root root  11M Sep 30 21:09 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.rev.1.bt2
-rw-r--r-- 1 root root 3.8M Sep 30 21:09 /biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.rev.2.bt2


# Profile the abundance of each gene in each sample

**Workflow**


1.   Map the reads against the bowtie2 database
2.   Calculate the coverage of each gene





In [15]:
import os
mgx_reads_dir = os.path.join(root_dir,"mgx_reads")
output_sam_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","mgx_gf_mapping","raw")
! mkdir -p {output_sam_dir}
sample_id="PSMB4MBK"
# p1 = os.path.join(mgx_reads_dir,sample_id+"_R1.fastq.gz")
# p2 = os.path.join(mgx_reads_dir,sample_id+"_R2.fastq.gz")
! ls -lh /content/drive/MyDrive/workshop_June2023/mgx_reads

# here we do subset:
# ! zcat /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_R1.fastq.gz | head -n 40000 > /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R1.fastq
# ! gzip /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R1.fastq
# ! zcat /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_R2.fastq.gz | head -n 40000 > /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R2.fastq
# ! gzip /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_subset_R2.fastq

p1 = os.path.join(mgx_reads_dir,sample_id+"_subset_R1.fastq.gz")
p2 = os.path.join(mgx_reads_dir,sample_id+"_subset_R2.fastq.gz")

! bowtie2 -x {demo_gene_call_dir}/nr.fa_bowtie2DB -1 {p1} -2 {p2} -S {output_sam_dir}/{sample_id}.sam # 22 min to finish on full / 3 min on subset

ls: cannot access '/content/drive/MyDrive/workshop_June2023/mgx_reads': No such file or directory
/bin/bash: line 1: bowtie2: command not found


### Profiling the coverage from the sam file

In [16]:
! which samtools
! samtools --version
! samtools coverage --help

/opt/conda/bin/samtools
samtools 1.22.1
Using htslib 1.22.1
Copyright (C) 2025 Genome Research Ltd.

Samtools compilation details:
    Features:       build=configure curses=yes 
    CC:             /opt/conda/conda-bld/samtools_1752527820713/_build_env/bin/aarch64-conda-linux-gnu-cc
    CPPFLAGS:       -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /opt/conda/include
    CFLAGS:         -Wall -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O3 -pipe -isystem /opt/conda/include -fdebug-prefix-map=/opt/conda/conda-bld/samtools_1752527820713/work=/usr/local/src/conda/samtools-1.22.1 -fdebug-prefix-map=/opt/conda=/usr/local/src/conda-prefix
    LDFLAGS:        -Wl,-O2 -Wl,--sort-common -Wl,--as-needed -Wl,-z,relro -Wl,-z,now -Wl,--allow-shlib-undefined -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib
    HTSDIR:         
    LIBS:           
    CURSES_LIB:     -ltinfow -lncursesw

HTSlib compilation details:
    Features:       build=configure libcurl=yes S3=

In [17]:
# sort the sam file
!samtools sort {output_sam_dir}/{sample_id}.sam -o {output_sam_dir}/{sample_id}.sorted_bam # 3 min
# generate the coverage file
!samtools coverage {output_sam_dir}/{sample_id}.sorted_bam > {output_sam_dir}/{sample_id}.coverage.txt

[E::hts_open_format] Failed to open file "/biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw/PSMB4MBK.sam" : No such file or directory
samtools sort: can't open "/biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw/PSMB4MBK.sam": No such file or directory
[E::hts_open_format] Failed to open file "/biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw/PSMB4MBK.sorted_bam" : No such file or directory
samtools coverage: Could not open "/biodata/resources/day3_lab4/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw/PSMB4MBK.sorted_bam": No such file or directory


In [18]:
# check the output:
! ls -lh /content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw
! head {output_sam_dir}/{sample_id}.coverage.txt

ls: cannot access '/content/drive/MyDrive/workshop_June2023/assembly_based_metagenomic_analysis/mgx_gf_mapping/raw': No such file or directory
